# Vector Database Fundamentals

**Module:** 02 — Vector Databases

Vector databases store embeddings and serve similarity search with metadata filters at scale.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain what a vector DB is and why keyword stores fall short
- Describe storage, indexes, collections, documents, metadata, filtering
- Design safe multi-tenant metadata schemas


## What / Why Vector Databases?

**Definition.** A **vector database** stores dense vectors and answers nearest-neighbor queries under latency, scale, and filter constraints.

**Why it matters.** GenAI apps embed content; at millions of vectors, scans miss interactive latency.

**How it works.** Embed → upsert vectors+payload → index → query embed → ranked IDs+metadata.

**Intuition.** A library shelved by meaning; filters are the guest list.

**Common pitfalls.**
- ANN ≠ exact
- Ignoring embedder versioning
- Skipping tenant filters

**When to use.** Semantic retrieval beyond toy corpora with filters/SLAs.

```mermaid
flowchart LR
 D[Docs]-->E[Embed]-->V[(Vector DB)]
 Q[Query]-->E2[Embed]-->S[Search]
 V-->S-->R[Top-k]-->L[LLM]
```
| System | Weakness for semantic search |
|--------|------------------------------|
| SQL | No native ANN |
| BM25 | Misses paraphrases |
| Vector DB | Not general OLTP |


In [ ]:
import time, numpy as np
rng=np.random.default_rng(0); dim=384
def topk(db,q,k=5):
    s=db@q; i=np.argpartition(-s,k)[:k]; return i[np.argsort(-s[i])]
for n in [1000,10000,50000]:
    db=rng.normal(size=(n,dim)).astype('float32'); db/=np.linalg.norm(db,axis=1,keepdims=True)+1e-9
    q=rng.normal(size=dim).astype('float32'); q/=np.linalg.norm(q)+1e-9
    t=time.perf_counter(); topk(db,q); print(f'N={n:,} {(time.perf_counter()-t)*1000:.2f} ms')


In [ ]:
import numpy as np
docs={'refund':np.array([.9,.8,.1,0]),'ship':np.array([.1,.1,.9,.8]),'pwd':np.array([0,.1,0,.9])}
for k,v in list(docs.items()): docs[k]=v/(np.linalg.norm(v)+1e-9)
q=np.array([.88,.78,.12,.02]); q/=np.linalg.norm(q)+1e-9
print(sorted(((float(docs[d]@q),d) for d in docs), reverse=True))


In [ ]:
import json
print('Authorization: Bearer YOUR_API_KEY')
print(json.dumps({'vector':[0.1,-0.2],'topK':5,'filter':{'tenant':{'$eq':'acme'}}},indent=2))


### Try it yourself — What / Why Vector Databases?

1. List 3 paraphrase queries BM25 misses.
2. Estimate when brute force fails for your N/QPS.


## Vector Storage

**Definition.** **Vector storage** persists float/quantized vectors plus payload bytes.

**Why it matters.** RAM/disk dominate cost; 100M×1536 float32 ≈ 600GB raw.

**How it works.** Segments/shards; optional compression + rescoring; payload inline or external.

**Intuition.** Vector = GPS; text = building fetched later.

**Common pitfalls.**
- float64 waste
- Mixed dimensions
- Unnormalized cosine

**When to use.** Decide format at collection design time.


In [ ]:
def gib(n,d,b=4,r=1,o=1.5): return n*d*b*r*o/(1024**3)
for n,d,rep in [(50000,384,1),(2_000_000,768,2)]:
    print(n, d, f'{gib(n,d,r=rep):.1f} GiB')


In [ ]:
import numpy as np
X=np.random.default_rng(1).normal(size=(5,8)).astype('float32'); Xu=X/(np.linalg.norm(X,axis=1,keepdims=True)+1e-9)
q=np.random.default_rng(2).normal(size=8); qu=q/(np.linalg.norm(q)+1e-9)
print(np.round(Xu@qu,3))


In [ ]:
from dataclasses import dataclass, field
@dataclass
class Rec:
    id:str; vector:object; metadata:dict=field(default_factory=dict)
print(Rec('a',[1,0,0],{'lang':'en'}))


### Try it yourself — Vector Storage

1. Compute GiB for your corpus.
2. Inline text vs object store—justify.


## Index

**Definition.** An **index** accelerates NN so you avoid full scans.

**Why it matters.** Latency grows with N without indexes; trade RAM/recall.

**How it works.** Graphs/IVF lists prune candidates; optional rescoring.

**Intuition.** Highways then side streets.

**Common pitfalls.**
- Untuned ef/nprobe
- Latency-only tuning

**When to use.** Interactive search beyond small N.

```mermaid
flowchart TD
 Q[Query]-->C[Candidates]-->R[Rescore]-->K[Top-k]
```


In [ ]:
def scan(n,d): return n*d
def ivf(n,d,nlist=1024,nprobe=16): return nlist*d + nprobe*max(n//nlist,1)*d
print('speedup', round(scan(1_000_000,768)/ivf(1_000_000,768),1),'x')


In [ ]:
import numpy as np
rng=np.random.default_rng(0); db=rng.normal(size=(2000,32)); db/=np.linalg.norm(db,axis=1,keepdims=True)+1e-9
q=db[0]; gold=set(np.argsort(-(db@q))[:10].tolist())
pool=rng.choice(len(db),50,False); approx=set(pool[np.argsort(-(db[pool]@q))[:10]].tolist())
print('toy recall', len(gold&approx)/10)


In [ ]:
print([('HNSW','efSearch'),('IVF','nprobe')])


### Try it yourself — Index

1. Set recall@10 and p95 targets.
2. What if you tune only latency?


## Collections

**Definition.** A **collection** is a named container with fixed dim/metric/schema.

**Why it matters.** Isolates embedders/envs/tenants; wrong isolation → garbage retrieval.

**How it works.** Create with name, dim, metric; ops target that collection.

**Intuition.** Typed table for vectors speaking one embedding language.

**Common pitfalls.**
- Mixing dims/models
- Prod used for experiments

**When to use.** Per model version / env / hard schema break.


In [ ]:
from dataclasses import dataclass
@dataclass
class Spec:
    name:str; dim:int; model:str
reg={'faq':Spec('faq',384,'bge-small')}
def ok(n,d,m):
    s=reg[n]; assert d==s.dim and m==s.model; print('OK')
ok('faq',384,'bge-small')


In [ ]:
import json
print(json.dumps({'vectors':{'size':384,'distance':'Cosine'}},indent=2))
print('api-key: YOUR_API_KEY')


In [ ]:
print([f'docs_{m}_{e}' for e in ['dev','prod'] for m in ['v1','v2']])


### Try it yourself — Collections

1. Name collections for (dev,prod)×(v1,v2).
2. One-line runbook for metric+dim.


## Documents

**Definition.** A **document**/point is id + vector + payload—usually a chunk.

**Why it matters.** Chunking/IDs/provenance drive RAG quality as much as indexes.

**How it works.** Split → embed → stable ids `doc#i` → copy parent metadata.

**Intuition.** Retrieve passages, not whole books.

**Common pitfalls.**
- Unstable IDs
- Chunks too big/small
- Lost source URIs

**When to use.** Design IDs before first ingest.


In [ ]:
def chunk(doc_id,text,size=8,overlap=2):
    w=text.split(); out=[]; i=idx=0
    while i<len(w):
        out.append((f'{doc_id}#{idx}',' '.join(w[i:i+size]))); idx+=1; i+=max(size-overlap,1)
    return out
print(chunk('p','Refunds within 30 days. Items unused.'))


In [ ]:
import json
print(json.dumps({'id':'p#0','values':[0.1,-0.2],'metadata':{'source':'kb.md','tenant':'acme'}},indent=2))


In [ ]:
print(chunk('p','Return window starts on delivery. Final sale on software.',size=5,overlap=2))


### Try it yourself — Documents

1. Chunk a FAQ two ways.
2. ID scheme resilient to re-chunking.


## Metadata

**Definition.** **Metadata**/payload: tenant, lang, time, ACL, source.

**Why it matters.** Similarity alone ≠ product-correct results.

**How it works.** Attach JSON at upsert; index filterable fields.

**Intuition.** Bouncer at the club door.

**Common pitfalls.**
- Secrets in payload
- Enum drift en/EN

**When to use.** Version a schema with the collection.


In [ ]:
REQ={'tenant':str,'lang':str,'source':str}
def validate(m):
    for k,t in REQ.items():
        assert k in m and isinstance(m[k],t)
    assert m['lang'] in {'en','es'}
    return True
print(validate({'tenant':'acme','lang':'en','source':'a.md'}))


In [ ]:
import numpy as np
rows=[('a',np.array([1.,0.]),{'tenant':'acme'}),('b',np.array([.9,.1]),{'tenant':'other'})]
q=np.array([1.,0.]); filt=[r for r in rows if r[2]['tenant']=='acme']
print(sorted(((float(r[1]@q),r[0]) for r in filt), reverse=True))


In [ ]:
print([('tenant','filter'),('title','display')])


### Try it yourself — Metadata

1. PII-safe schema for multi-tenant help center.


## Filtering

**Definition.** **Filtering** applies metadata predicates during/after ANN.

**Why it matters.** Prevents tenant leaks and stale/wrong-language hits.

**How it works.** Pre-filter vs post-filter; selective filters need over-fetch or filter-aware ANN.

**Intuition.** Similarity proposes; filters dispose.

**Common pitfalls.**
- Post-filter + tiny top_k → empty
- Client-supplied tenant

**When to use.** Always tenant/ACL in multi-tenant apps.

```mermaid
flowchart TD
 Q[Query+filter]-->P{Strategy}
 P -->|pre|A[ANN allowed]
 P -->|post|B[ANN then predicate]
 A-->O[Top-k]; B-->O
```


In [ ]:
import numpy as np
rng=np.random.default_rng(0); N=1000; X=rng.normal(size=(N,16)); X/=np.linalg.norm(X,axis=1,keepdims=True)+1e-9
tenant=np.array(['acme' if i%20==0 else 'other' for i in range(N)]); q=X[0]
def search(M):
    order=np.argsort(-(X@q))[:M]; return [i for i in order if tenant[i]=='acme'][:5]
print('M10',search(10),'M200',search(200))


In [ ]:
print({'pinecone':{'tenant':{'$eq':'acme'}},'qdrant':{'must':[{'key':'tenant','match':{'value':'acme'}}]}})


In [ ]:
class Guard:
    def __init__(self,t): self.t=t
    def inject(self,f=None):
        b={'tenant':{'$eq':self.t}}; return {'$and':[b,f]} if f else b
print(Guard('acme').inject({'lang':{'$eq':'en'}}))


### Try it yourself — Filtering

1. Pre vs post filter for 1% tenant.
2. Write tenant∧(lang)∧freshness filter.


## Glossary

- **collection**: Named vector container
- **payload**: Metadata on a point
- **ANN**: Approximate NN


## Summary & Key Takeaways

- Vector DBs = fast filterable similarity.
- Collections bind dim/metric/model.
- Metadata makes neighbors product-correct.

### Practice

Design collection+metadata for a 200k bilingual two-tenant wiki.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
